# Выбросы
Если у тебя есть статистическая задача, в которое ты использушь неробастные оценкки (на которые влияют выбросы), например, среднее или T-test.

**Проверь выбросы!**

Сначала визуально boxplot, если они визульно есть, то стоит их отбросить по IQR или тп.

**Обращай внимание на смысл данных** не может быть -5 апельсинов.

In [2]:
import pandas as pd

def IQR_outliers_remove(df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    for column in columns:
        Q1 = df[column].quantile(0.25)
        Q3 = df[column].quantile(0.75)
        IQR = Q3 - Q1
        df = df[(df[column] >= Q1 - 1.5 * IQR) & (df[column] <= Q3 + 1.5 * IQR)]
    return df

# Корреляция

1. Проверка типов переменных
- В функу передаем categorial - список категориальных переменных.
- проверка нормальности - тест Шапиро

2. Выбор метрики связи
- Номинальная + номинальная → χ² (преобразуем значение в [-1, 1] по степени зависимости, например Cramér's V).
- Колич. (норм.) + колич. (норм.) → корреляция Пирсона.
- Колич. (ненорм.) + колич. (ненорм.) → корреляция Спирмена.
- Номинальная + количественная (или разн. шкалы) → используем отношение дисперсий (ANOVA), и нормируем в диапазон [-1, 1].

<img src="Тип%20корреляции.jpg">

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr, f_oneway, chi2_contingency
from scipy.stats import shapiro

def is_normal(series, alpha=0.05):
    if series.nunique() < 3:
        return False  # слишком мало уникальных значений
    stat, p = shapiro(series.dropna())
    return p > alpha

def cramers_v(x, y):
    confusion_matrix = pd.crosstab(x, y)
    chi2 = chi2_contingency(confusion_matrix)[0]
    n = confusion_matrix.sum().sum()
    phi2 = chi2 / n
    r, k = confusion_matrix.shape
    phi2corr = max(0, phi2 - ((k-1)*(r-1))/(n-1))
    rcorr = r - ((r-1)**2)/(n-1)
    kcorr = k - ((k-1)**2)/(n-1)
    return np.sqrt(phi2corr / min((kcorr-1), (rcorr-1)))

def normalized_variance_ratio(x, y):
    # ANOVA F-test
    try:
        groups = [x[y == cat] for cat in np.unique(y)]
        f_stat, p_val = f_oneway(*groups)
        if np.isnan(f_stat): return 0
        # Нормализация значения F в диапазон от -1 до 1 (грубо, но для визуализации допустимо)
        norm_value = (f_stat / (f_stat + 1)) * 2 - 1  # в диапазон [0, 1] -> [-1, 1]
        return norm_value
    except:
        return 0

def plot_correlation_heatmap(df, categorial):
    cols = df.columns
    result = pd.DataFrame(index=cols, columns=cols, dtype=float)

    for i in range(len(cols)):
        for j in range(len(cols)):
            col1 = df[cols[i]]
            col2 = df[cols[j]]

            if cols[i] in categorial and cols[j] in categorial:
                # χ² → Cramér's V
                val = cramers_v(col1, col2)

            elif cols[i] not in categorial and cols[j] not in categorial:
                # Колич. + Колич.
                if is_normal(col1) and is_normal(col2):
                    val = pearsonr(col1, col2)[0]
                else:
                    val = spearmanr(col1, col2)[0]

            else:
                # Смешанные шкалы → отношение дисперсий
                if cols[i] in categorial:
                    val = normalized_variance_ratio(df[cols[j]], col1)
                else:
                    val = normalized_variance_ratio(df[cols[i]], col2)

            result.iloc[i, j] = val

    # Построение хитмапы
    plt.figure(figsize=(12, 10))
    sns.heatmap(result.astype(float), annot=True, cmap="coolwarm", center=0, fmt=".2f")
    plt.title("Корреляционная тепловая карта (автоматический метод)")
    plt.show()

# Графики выбросов

**Если нужно графиком показать кол-во выбросов**

In [ ]:
import missingno as msno

msno.bar(df)

**Корреляция между пропусками**

In [ ]:
msno.heatmap(df)

# Нормальность

Изучая нормальность данных нужно показать и гистограмму, и боксплот.
В `base.ipynp` есть функция `norm()`, которая выводит нормально ли распределены данные

<img src="Предобработка_1.png">
<img src="Предобработка_2.png">

# Максимум в задаче с моделированием
Если в задаче про моделирование есть пункт **найдите максимум**, то ты должен найти оптимальные параметры задачи, а потом с ними с моделировть лучший вариант из всех возможных

# Указание параметров, генерирующих распределения
По опыту прошлого года за это рубили баллы, поэтому если у вас есть какое-то сгенерированое вами распределение, то не поленитесь и покажите параметры, которые его задают.

# Читаемые графики
Указывайте заголовок, название осей, x-lim, y-lim. 

**Проверяйте свой график на "читаемость"**

# Задача на симуляцию. Ошибка с округлением
В задаче на симуляцию после того как вы получили оптимум, прогоните еще раз но с сильно меньшим диапазоном, например, +- 5, но с сильно большим количеством итераций и меньшим шагом, чтобы точно не ошибиться

# Задача на оценку
Где просят оценить интервал, подставлять максимальные и минимальные значения из условия